# 片上单元测试 (On-Chip Unit Test)

复用 `golden_module_tb.py` 生成向量，在 FPGA 板上跑 module-level 测试。

**默认数据通路 (staging=hbm, Host 仅 HBM + INST_BRAM):**
```
Host → HBM + INST_BRAM
       → CDMA → 片内计算 → CDMA → HBM → Host 读回比对
```

Lab 回退：`staging=preload` 直写片内 + scatter 读 obuf。

环境: conda `chip_test_env`，工具 `tests/xdma_exe/xdma_rw.exe`

## 0. 环境检查

In [1]:
import sys
from pathlib import Path

# 确保 unit-tb 目录在路径中
UNIT_TB_DIR = Path(".").resolve()
if str(UNIT_TB_DIR) not in sys.path:
    sys.path.insert(0, str(UNIT_TB_DIR))

REPO_ROOT = UNIT_TB_DIR.parent.parent.parent
print(f"Repo root: {REPO_ROOT}")
print(f"Unit-TB dir: {UNIT_TB_DIR}")

Repo root: E:\work2026\runnan_xu\FPGA\EdgeYOLO-FPGA
Unit-TB dir: E:\work2026\runnan_xu\FPGA\EdgeYOLO-FPGA\tests\chip\unit-tb


In [2]:
from xdma_win import XDMAWin, XDMA_RW_EXE, REGS_BASE, REG_DECODER_STATUS

# 验证 xdma_rw.exe 存在
assert XDMA_RW_EXE.exists(), f"xdma_rw.exe not found: {XDMA_RW_EXE}"
print(f"xdma_rw.exe: {XDMA_RW_EXE} ✓")

# 快速通信测试: 读 DECODER_STATUS 寄存器
xdma = XDMAWin(verbose=True)
status = xdma.read_u32(REGS_BASE + REG_DECODER_STATUS)
print(f"\nDECODER_STATUS = 0x{status:08x}")
print("XDMA 通信正常 ✓" if status != 0xDEADBEEF else "WARNING: 读到异常值")

xdma_rw.exe: E:\work2026\runnan_xu\FPGA\EdgeYOLO-FPGA\tests\xdma_exe\xdma_rw.exe ✓
  [xdma] E:\work2026\runnan_xu\FPGA\EdgeYOLO-FPGA\tests\xdma_exe\xdma_rw.exe c2h_0 read 0x105000040 -b -f C:\Users\zczho\AppData\Local\Temp\tmp16e037vd.bin -l 4

DECODER_STATUS = 0x00000002
XDMA 通信正常 ✓


## 1. 生成测试数据

使用 `golden_module_tb.py` 为指定 case 生成完整测试向量。

可选 case 列表:
- `dcim_matmul`: DCIM 矩阵乘 (dcim_tiny_1x1, conv3_s2_c32_to64, ...)
- `qa`: 量化单元 (qa_c16_signed, qa_c64_clip, ...)
- `dqa`: 反量化单元 (dqa_c16_small, dqa_c32_mid, ...)
- `im2col`: im2col 变换
- `conv_pipeline`: 单层 conv 全链路 (im2col→CDMA→DCIM→DQA→QA)
- `mini_network`: 多层网络 (2-3 conv + residual)

In [3]:
from gen_data import generate_case, list_cases, list_modules

# 查看所有可用模块
print("Available modules:", list_modules())

Available modules: ['dcim_matmul', 'dqa', 'qa', 'im2col', 'mp', 'us', 'add', 'conv_pipeline', 'mini_network', 'concat_by_cdma', 'cdma_memtest']


In [4]:
# 查看某个模块的所有可用 variant
MODULE_CASE = "dcim_matmul"  # ← 修改这里选择模块
variants = list_cases(MODULE_CASE)
print(f"\n{MODULE_CASE} variants:")
for v in variants:
    print(f"  • {v}")


dcim_matmul variants:
  • dcim_tiny_1x1
  • conv6_s2_c3_to16
  • conv3_s2_c32_to64
  • conv1_c64_to32
  • conv3_c128_to128
  • int16_tiny_1x1
  • int16_conv3_c32_c64
  • int16_conv1_c128
  • extreme_int8_1x1_c512_to512
  • extreme_int8_3x3_c128_to512
  • extreme_int8_6x6_c3_to64
  • extreme_int16_3x3_c128
  • dcim_model_0_conv  M=25600 K=108 N=16 acc=2 in=320x320
  • dcim_model_1_conv  M=6400 K=144 N=32 acc=3 in=160x160
  • dcim_model_2_cv1_conv  M=6400 K=32 N=16 acc=1 in=80x80
  • dcim_model_2_cv2_conv  M=6400 K=32 N=16 acc=1 in=80x80
  • dcim_model_2_cv3_conv  M=6400 K=32 N=32 acc=1 in=80x80
  • dcim_model_2_m_0_cv1_conv  M=6400 K=16 N=16 acc=1 in=80x80
  • dcim_model_2_m_0_cv2_conv  M=6400 K=144 N=16 acc=3 in=80x80
  • dcim_model_3_conv  M=1600 K=288 N=64 acc=5 in=80x80
  • dcim_model_4_cv1_conv  M=1600 K=64 N=32 acc=1 in=40x40
  • dcim_model_4_cv2_conv  M=1600 K=64 N=32 acc=1 in=40x40
  • dcim_model_4_cv3_conv  M=1600 K=64 N=64 acc=1 in=40x40
  • dcim_model_4_m_0_cv1_conv  M=1600 

In [5]:
# ═══════════════════════════════════════════════════════════════════
# 配置: 选择要测试的 case
# ═══════════════════════════════════════════════════════════════════
MODULE_CASE    = "dcim_matmul"     # 模块类型
MODULE_VARIANT = "dcim_tiny_1x1"   # 具体用例 (最小规模, 适合首次验证)
QUANT          = "int8"            # 量化模式: int8 / int16
# ═══════════════════════════════════════════════════════════════════

run_dir = generate_case(MODULE_CASE, MODULE_VARIANT, quant=QUANT)
print(f"\n生成完成! 运行目录: {run_dir}")
print("\n文件列表:")
for f in sorted(run_dir.iterdir()):
    print(f"  {f.name:25s} ({f.stat().st_size:>8,} bytes)")

[gen] Generating: dcim_matmul/dcim_tiny_1x1 (quant=int8)
[gen] Output dir: E:\work2026\runnan_xu\FPGA\EdgeYOLO-FPGA\tests\chip\unit-tb\runs\dcim_matmul_dcim_tiny_1x1_qint8
[gen] Generated module_tb: module=dcim_matmul case=dcim_tiny_1x1 layer=model.2.cv1.conv
  shape=[int8] M=4 K=32 N=16 acc_depth=1 in_hw=2x2(manual) words=128 out_dir=E:\work2026\runnan_xu\FPGA\EdgeYOLO-FPGA\tests\chip\unit-tb\runs\dcim_matmul_dcim_tiny_1x1_qint8
[gen] Done. Files: ['act.hex', 'checks.txt', 'expected.hex', 'fast_preload.svh', 'hbm_image.hex', 'inst.hex', 'manifest.txt', 'module_manifest.svh', 'preload.txt', 'wb_init.hex', 'weight_tile0.hex']

生成完成! 运行目录: E:\work2026\runnan_xu\FPGA\EdgeYOLO-FPGA\tests\chip\unit-tb\runs\dcim_matmul_dcim_tiny_1x1_qint8

文件列表:
  act.hex                   (     544 bytes)
  checks.txt                (      43 bytes)
  expected.hex              (   4,352 bytes)
  fast_preload.svh          (   1,491 bytes)
  hbm_image.hex             (1,116,288 bytes)
  inst.hex              

## 2. 检查生成的数据

In [6]:
# 查看 preload.txt: 需要上传到 FPGA 的数据
print("=== preload.txt ===")
print((run_dir / "preload.txt").read_text())

print("\n=== checks.txt ===")
print((run_dir / "checks.txt").read_text())

print("\n=== inst.hex (前 10 行) ===")
lines = (run_dir / "inst.hex").read_text().splitlines()
print(f"总指令数: {len(lines)} words")
for l in lines[:10]:
    print(f"  {l}")

=== preload.txt ===
act.hex 0000000100000000
weight_tile0.hex 0000000100040000


=== checks.txt ===
dcim_tiny_1x1 expected.hex 800000 128 0 4


=== inst.hex (前 10 行) ===
总指令数: 42 words
  90000060
  00000004
  00000106
  00000001
  00000000
  00000000
  00000004
  00000004
  00000000
  00004000


## 3. 上传 HBM 并下发指令

Host 写 `hbm_image.hex`（含 wb 嵌入）到 HBM；`inst.hex` 经 HBM 补丁（input CDMA + output drain）写 INST_BRAM。

Lab 回退：`runner.upload_preload(run_dir)` + `upload_inst_raw()`。

In [7]:
from xdma_win import ChipRunnerWin

runner = ChipRunnerWin(verbose=True)
runner.upload_hbm(run_dir)

[preload] act.hex (256 bytes) -> 0x0100000000
  [xdma] E:\work2026\runnan_xu\FPGA\EdgeYOLO-FPGA\tests\xdma_exe\xdma_rw.exe h2c_0 write 0x100000000 -b -f C:\Users\zczho\AppData\Local\Temp\tmppvbjxchs.bin -l 256
[preload] weight_tile0.hex (1024 bytes) -> 0x0100040000
  [xdma] E:\work2026\runnan_xu\FPGA\EdgeYOLO-FPGA\tests\xdma_exe\xdma_rw.exe h2c_0 write 0x100040000 -b -f C:\Users\zczho\AppData\Local\Temp\tmp1292awfw.bin -l 1024


In [8]:
n_words = runner.upload_inst(run_dir)
print(f"\n指令已上传: {n_words} words")

[inst] 42 words (168 bytes) -> INST_BRAM 0x0104000000
  [xdma] E:\work2026\runnan_xu\FPGA\EdgeYOLO-FPGA\tests\xdma_exe\xdma_rw.exe h2c_0 write 0x104000000 -b -f C:\Users\zczho\AppData\Local\Temp\tmpt1b8ur47.bin -l 168

指令已上传: 42 words


## 4. 执行并等待完成

In [9]:
# Step 3: 启动 INST_Decoder
runner.start_decoder(n_words)

# Step 4: 等待执行完成
runner.poll_done(timeout_s=120.0)
print("\n执行完成!")

[obuf] clearing 8 tiles x 256 B (px=4, wpt=4)
  [xdma] E:\work2026\runnan_xu\FPGA\EdgeYOLO-FPGA\tests\xdma_exe\xdma_rw.exe h2c_0 write 0x101000000 -b -f C:\Users\zczho\AppData\Local\Temp\tmpzrffg91z.bin -l 256
  [xdma] E:\work2026\runnan_xu\FPGA\EdgeYOLO-FPGA\tests\xdma_exe\xdma_rw.exe h2c_0 write 0x101040000 -b -f C:\Users\zczho\AppData\Local\Temp\tmpk0wralon.bin -l 256
  [xdma] E:\work2026\runnan_xu\FPGA\EdgeYOLO-FPGA\tests\xdma_exe\xdma_rw.exe h2c_0 write 0x101080000 -b -f C:\Users\zczho\AppData\Local\Temp\tmp0y75w2f9.bin -l 256
  [xdma] E:\work2026\runnan_xu\FPGA\EdgeYOLO-FPGA\tests\xdma_exe\xdma_rw.exe h2c_0 write 0x1010c0000 -b -f C:\Users\zczho\AppData\Local\Temp\tmp8sw6ewo7.bin -l 256
  [xdma] E:\work2026\runnan_xu\FPGA\EdgeYOLO-FPGA\tests\xdma_exe\xdma_rw.exe h2c_0 write 0x101100000 -b -f C:\Users\zczho\AppData\Local\Temp\tmpnsoa3krf.bin -l 256
  [xdma] E:\work2026\runnan_xu\FPGA\EdgeYOLO-FPGA\tests\xdma_exe\xdma_rw.exe h2c_0 write 0x101140000 -b -f C:\Users\zczho\AppData\Loca

## 5. 读回结果并比对 Golden

Step 4 完成后，从 HBM `0x100000` 线性读回（与 `expected.hex` 逐 word 比对）。

In [10]:
# Step 5: 从 HBM 读回结果, 与 expected.hex 逐 word 比对
# 前提: 已跑完 Step 2~4 (hbm + inst + decoder DONE)
results = runner.read_check(run_dir, from_hbm=True)

print("\n" + "="*60)
print("Results Summary")
print("="*60)
all_pass = True
for r in results:
    status = "PASS" if r["pass"] else "FAIL"
    print(f"  {r['name']:30s} {status}  ({r['passed']}/{r['total_words']} words)")
    if not r["pass"]:
        all_pass = False
        m = r["first_mismatch"]
        print(f"    First mismatch at word {m['word']}:")
        print(f"      expected: {m['expected']}")
        print(f"      got:      {m['got']}")

print("\n" + ("ALL PASS" if all_pass else "SOME FAILED"))

[check] 'dcim_tiny_1x1': scatter read 128 words from tile_obuf (wpt=4)
  [xdma] E:\work2026\runnan_xu\FPGA\EdgeYOLO-FPGA\tests\xdma_exe\xdma_rw.exe c2h_0 read 0x101000000 -b -f C:\Users\zczho\AppData\Local\Temp\tmpf1qp8bf_.bin -l 256
  [xdma] E:\work2026\runnan_xu\FPGA\EdgeYOLO-FPGA\tests\xdma_exe\xdma_rw.exe c2h_0 read 0x101040000 -b -f C:\Users\zczho\AppData\Local\Temp\tmpu2awojv3.bin -l 256
  [xdma] E:\work2026\runnan_xu\FPGA\EdgeYOLO-FPGA\tests\xdma_exe\xdma_rw.exe c2h_0 read 0x101080000 -b -f C:\Users\zczho\AppData\Local\Temp\tmp9encztac.bin -l 256
  [xdma] E:\work2026\runnan_xu\FPGA\EdgeYOLO-FPGA\tests\xdma_exe\xdma_rw.exe c2h_0 read 0x1010c0000 -b -f C:\Users\zczho\AppData\Local\Temp\tmpr6_v31xg.bin -l 256
  [xdma] E:\work2026\runnan_xu\FPGA\EdgeYOLO-FPGA\tests\xdma_exe\xdma_rw.exe c2h_0 read 0x101100000 -b -f C:\Users\zczho\AppData\Local\Temp\tmpnj6i6n21.bin -l 256
  [xdma] E:\work2026\runnan_xu\FPGA\EdgeYOLO-FPGA\tests\xdma_exe\xdma_rw.exe c2h_0 read 0x101140000 -b -f C:\Users

## 6. 一键执行 (完整流程)

以上 Step 1~5 封装为 `run_case()` 一键调用:

In [11]:
from xdma_win import ChipRunnerWin
from gen_data import generate_case

# 一键: 生成 + 上传 + 执行 + 比对
run_dir = generate_case("dcim_matmul", "dcim_tiny_1x1", quant="int8")
runner = ChipRunnerWin(verbose=True)
results = runner.run_case(run_dir, timeout_s=120.0, staging="hbm")

[gen] Generating: dcim_matmul/dcim_tiny_1x1 (quant=int8)
[gen] Output dir: E:\work2026\runnan_xu\FPGA\EdgeYOLO-FPGA\tests\chip\unit-tb\runs\dcim_matmul_dcim_tiny_1x1_qint8
[gen] Generated module_tb: module=dcim_matmul case=dcim_tiny_1x1 layer=model.2.cv1.conv
  shape=[int8] M=4 K=32 N=16 acc_depth=1 in_hw=2x2(manual) words=128 out_dir=E:\work2026\runnan_xu\FPGA\EdgeYOLO-FPGA\tests\chip\unit-tb\runs\dcim_matmul_dcim_tiny_1x1_qint8
[gen] Done. Files: ['act.hex', 'checks.txt', 'expected.hex', 'fast_preload.svh', 'hbm_image.hex', 'inst.hex', 'manifest.txt', 'module_manifest.svh', 'preload.txt', 'wb_init.hex', 'weight_tile0.hex']

Running case: dcim_matmul_dcim_tiny_1x1_qint8 (staging=preload)
[preload] act.hex (256 bytes) -> 0x0100000000
  [xdma] E:\work2026\runnan_xu\FPGA\EdgeYOLO-FPGA\tests\xdma_exe\xdma_rw.exe h2c_0 write 0x100000000 -b -f C:\Users\zczho\AppData\Local\Temp\tmpbui4jcws.bin -l 256
[preload] weight_tile0.hex (1024 bytes) -> 0x0100040000
  [xdma] E:\work2026\runnan_xu\FPGA\

## 7. 批量测试

复用 module_tb 的 smoke suite 定义, 逐个执行并汇总:

In [12]:
from xdma_win import ChipRunnerWin, case_timeout_s
from gen_data import generate_case

# ═══════════════════════════════════════════════════════════════════
# 批量配置 (默认 staging=hbm, Host 仅读写 HBM+INST_BRAM)
# ═══════════════════════════════════════════════════════════════════
BATCH = [
    # (module_case, variant, quant)
    ("dcim_matmul", "dcim_tiny_1x1", "int8"),
    ("dcim_matmul", "conv6_s2_c3_to16", "int8"),
    ("dcim_matmul", "conv3_s2_c32_to64", "int8"),
    ("qa", "qa_c16_signed", "int8"),
    ("dqa", "dqa_c16_small", "int8"),
    ("im2col", "im2col_6x6_s2_c3", "int8"),
]
STOP_ON_FAIL = False
STAGING = "hbm"  # lab fallback: "preload"
# ═══════════════════════════════════════════════════════════════════

runner = ChipRunnerWin(verbose=False)
summary = []

for case, variant, quant in BATCH:
    try:
        run_dir = generate_case(case, variant, quant=quant)
        tmo = case_timeout_s(case, variant)
        results = runner.run_case(run_dir, timeout_s=tmo, staging=STAGING)
        passed = all(r["pass"] for r in results)
        summary.append((f"{case}/{variant}", "PASS" if passed else "FAIL", results))
        if not passed and STOP_ON_FAIL:
            break
    except Exception as e:
        summary.append((f"{case}/{variant}", f"ERROR: {e}", []))
        if STOP_ON_FAIL:
            break

print("\n" + "="*70)
print("Batch Results")
print("="*70)
n_pass = 0
for name, status, _ in summary:
    icon = "✓" if status == "PASS" else "✗"
    print(f"  {icon} {name:40s} {status}")
    if status == "PASS":
        n_pass += 1

print(f"\nTotal: {n_pass}/{len(summary)} PASS")

[gen] Generating: dcim_matmul/dcim_tiny_1x1 (quant=int8)
[gen] Output dir: E:\work2026\runnan_xu\FPGA\EdgeYOLO-FPGA\tests\chip\unit-tb\runs\dcim_matmul_dcim_tiny_1x1_qint8
[gen] Generated module_tb: module=dcim_matmul case=dcim_tiny_1x1 layer=model.2.cv1.conv
  shape=[int8] M=4 K=32 N=16 acc_depth=1 in_hw=2x2(manual) words=128 out_dir=E:\work2026\runnan_xu\FPGA\EdgeYOLO-FPGA\tests\chip\unit-tb\runs\dcim_matmul_dcim_tiny_1x1_qint8
[gen] Done. Files: ['act.hex', 'checks.txt', 'expected.hex', 'fast_preload.svh', 'hbm_image.hex', 'inst.hex', 'manifest.txt', 'module_manifest.svh', 'preload.txt', 'wb_init.hex', 'weight_tile0.hex']
[gen] Generating: dcim_matmul/conv6_s2_c3_to16 (quant=int8)
[gen] Output dir: E:\work2026\runnan_xu\FPGA\EdgeYOLO-FPGA\tests\chip\unit-tb\runs\dcim_matmul_conv6_s2_c3_to16_qint8
[gen] Generated module_tb: module=dcim_matmul case=conv6_s2_c3_to16 layer=model.0.conv
  shape=[int8] M=36 K=108 N=16 acc_depth=2 in_hw=12x12(manual) words=1152 out_dir=E:\work2026\runnan_x

## 8. 调试工具

当测试失败时, 可用以下工具进行诊断:

In [13]:
import numpy as np
from xdma_win import (
    XDMAWin, hex_to_bin,
    REGS_BASE, INST_BASE, VPU_BUF_BASE, TILE_IBUF_BASE, TILE_OBUF_BASE,
    REG_STATUS, REG_DECODER_CTRL, REG_INST_COUNT, REG_DECODER_STATUS,
)

xdma = XDMAWin(verbose=True)

# 读取所有控制寄存器
print("=== VPU_AXI_Regs ===")
print(f"  STATUS         (0x04) = 0x{xdma.read_u32(REGS_BASE + 0x04):08x}")
print(f"  DECODER_CTRL   (0x38) = 0x{xdma.read_u32(REGS_BASE + 0x38):08x}")
print(f"  INST_COUNT     (0x3C) = 0x{xdma.read_u32(REGS_BASE + 0x3C):08x}")
print(f"  DECODER_STATUS (0x40) = 0x{xdma.read_u32(REGS_BASE + 0x40):08x}")

ImportError: cannot import name 'REG_STATUS' from 'xdma_win' (E:\work2026\runnan_xu\FPGA\EdgeYOLO-FPGA\tests\chip\unit-tb\xdma_win.py)

In [ ]:
# 手动读取任意 FPGA 地址 (用于 debug)
READ_ADDR = TILE_OBUF_BASE  # ← 修改地址
READ_WORDS = 4               # ← 读多少个 128-bit word

raw = xdma.read(READ_ADDR, READ_WORDS * 16)
for i in range(READ_WORDS):
    word = raw[i*16:(i+1)*16]
    print(f"  [{i:3d}] {word.hex()}")

In [ ]:
# 手动写入单个 32-bit 寄存器 (用于 debug)
# xdma.write_u32(REGS_BASE + REG_DECODER_CTRL, 0)  # 取消注释执行

## 9. L4 cout-tiling 验证

验证 **model.7.conv (128→256)** 的两个 pass，确认 `out_ch_offset` 机制正确：
- **tile0**: weights[0:128]  → output ch[0:128]
- **tile1**: weights[128:256] → output ch[128:255]
- **合并结果** 与 numpy golden 对比

运行前请确认 FPGA 已上电并正常响应。

In [ ]:
"""
cout-tiling 单算子验证：model.7.conv (128→256, k=3, s=2)
分两个 pass 各跑 128 output channels，对比 numpy golden。
"""
import sys, numpy as np
from pathlib import Path
sys.path.insert(0, '.')
sys.path.insert(0, '../../../rtl/tb/lite_bd/module_tb')
sys.path.insert(0, '../../../tools')

from gen_data import generate_case
from xdma_win import ChipRunnerWin, VPU_BUF_BASE
from golden_module_tb import (
    make_conv_pipeline_case, load_network, conv_meta, write_inst, write_hex,
    alloc_flat, int8_hwc_words, OBUF_PHY_BASE,
)

NETWORK_JSON = '../../../model/yolov5n/parsed/network.json'
net = load_network(NETWORK_JSON)
runner = ChipRunnerWin(verbose=False)

# 固定同一随机输入，确保两个 pass 看到相同激活
rng = np.random.default_rng(7)
H, W = 4, 4
feat_in = rng.integers(-128, 128, (H, W, 128), dtype=np.int16).astype(np.int8)

results = {}
for tile_idx, offset in enumerate([0, 128]):
    case_name = f"pipe_model7_tile{tile_idx}"
    run_dir = Path(f"./runs/cout_tiling/{case_name}")
    run_dir.mkdir(parents=True, exist_ok=True)

    spec = {
        "name": case_name,
        "layer": "model.7.conv",
        "in_hw": (H, W),
        "out_ch_limit": 128,
        "out_ch_offset": offset,
    }
    meta_dict = make_conv_pipeline_case(str(run_dir), net, spec,
                                        np.random.default_rng(0), feat=feat_in)
    write_inst(str(run_dir / "inst.hex"), meta_dict["fast_inst"])

    print(f"\n=== tile{tile_idx} (ch[{offset}:{offset+128}]) ===")
    print(f"  shape: {meta_dict['shape']}")
    tile_results = runner.run_case(run_dir, staging="hbm")
    passed = all(r.get("pass", False) for r in tile_results)
    pw = sum(r.get("pass_words", 0) for r in tile_results)
    tw = sum(r.get("total_words", 0) for r in tile_results)
    status = "PASS" if passed else "FAIL"
    print(f"  {status} {pw}/{tw} words")
    results[f"tile{tile_idx}"] = {"pass": passed, "pw": pw, "tw": tw}

print("\n=== 汇总 ===")
all_pass = all(v["pass"] for v in results.values())
print("  两 pass 全 PASS" if all_pass else "  有 pass 失败，检查上方输出")
print("  model.7.conv cout-tiling 验证", "✓ DONE" if all_pass else "✗ NEED DEBUG")

## 10. L4 完整网络渐进测试

使用 `_l4_full_network_test.py` 实现的 `run_yolov5n_backbone_neck`，
可以按 `--stop-at` 逐层推进：

```
# 命令行（在 tests/chip/unit-tb 目录下）
python _l4_full_network_test.py --stop-at model.3.conv          # 前 3 层
python _l4_full_network_test.py --stop-at model.7.conv          # 到 backbone 末
python _l4_full_network_test.py                                  # 完整 Backbone+Neck
```

或在本 Cell 中直接调用：

In [ ]:
"""
L4 完整网络渐进测试（Backbone + Neck）
修改 STOP_AT 控制测到哪层停止；设为 "" 跑全程。
"""
import sys, numpy as np
from pathlib import Path
sys.path.insert(0, '.')
sys.path.insert(0, '../../../rtl/tb/lite_bd/module_tb')
sys.path.insert(0, '../../../tools')

# 导入网络测试主函数
import importlib, importlib.util
spec = importlib.util.spec_from_file_location(
    "_l4_full_network_test", "./_l4_full_network_test.py")
l4 = importlib.util.module_from_spec(spec)
spec.loader.exec_module(l4)

# ── 配置 ─────────────────────────────────────
STOP_AT  = "model.5.conv"   # ← 修改这里，或设为 "" 跑全部
DRY_RUN  = False            # True: 只跑 numpy, 不上 FPGA
SEED     = 42
VERBOSE  = False
# ──────────────────────────────────────────────

runner = None if DRY_RUN else l4.ChipRunnerWin()

rng = np.random.default_rng(SEED)
img_int8 = rng.integers(-128, 128, (320, 320, 3), dtype=np.int16).astype(np.int8)
print(f"输入: 320×320×3 INT8 | stop_at='{STOP_AT}' | dry_run={DRY_RUN}")

acts = l4.run_yolov5n_backbone_neck(
    runner, img_int8,
    dry_run=DRY_RUN,
    stop_at=STOP_AT,
    verbose=VERBOSE,
)

print("\n各层输出形状：")
for k, v in acts.items():
    if v is not None:
        print(f"  {k:38s}: {v.shape}")